# Object-Oriented Programming in Python:
# Game Entities, Inheritance, Polymorphism and the Object Model
**Software Development & Python · 90-minute practical seminar**

Work in pairs: **predict → run → modify → explain**. We evolve a small game model,
not a complete game engine. Use Python **3.10+** and the standard library only.
Run cells in order; restart the kernel when replaying the whole lab. Some classes
are deliberately redefined as the design improves: an existing instance keeps its
original class, so we recreate examples after a refactoring.

You already know syntax, functions, containers, classes, and the introductory object
model. Today ask: **What problem does each mechanism solve?**

| Section | Core minutes |
| --- | ---: |
| 1. Warm-up | 3 |
| 2. Encapsulation + exercise | 9 |
| 3. Inheritance | 5 |
| 4. Overriding and super() | 5 |
| 5. Polymorphism | 5 |
| 6. Duck typing | 4 |
| 7. Abstract base classes | 6 |
| 8. Multiple inheritance + mixin exercise | 6 |
| 9. MRO | 5 |
| 10. Composition | 6 |
| 11. Dataclasses | 6 |
| 12. Slots | 4 |
| 13. Class methods | 3 |
| 14. Special methods | 3 |
| 15. Attribute lookup + logging exercise | 6 |
| 16. Final challenge | 12 |
| 17. Summary | 2 |
| **Total (excluding OPTIONAL material)** | **90** |

OPTIONAL cells are extension material; skip them on the core route. Predictions and
discussion questions are oral checkpoints. The four core coding exercises have
workspaces below and instructor solutions at the very end. Expected errors in demos
are caught so **Run All** completes even before students implement the exercises.

## 1. Warm-up: A Simple Entity · 3 min
A game needs independent health for each character. An `Entity` class describes
construction and behavior; `player` and `enemy` are instances. `self` receives the
instance on which the method is called. `name` and `hp` are instance attributes.

**Predict:** after the hit, does the enemy's health change too?

In [1]:
class Entity:
    def __init__(self, name, hp):
        self.name = name
        self.hp = hp

    def take_damage(self, amount):
        self.hp -= amount

In [2]:
player = Entity("Hero", 100)
enemy = Entity("Goblin", 30)
player.take_damage(12)
print(player.name, player.hp)
print(enemy.name, enemy.hp)
print(player.__dict__)

Hero 88
Goblin 30
{'name': 'Hero', 'hp': 88}


**Discuss:** in `player.take_damage(12)`, what supplies `self`? Each instance has
its own state even though both use the same method implementation.

## 2. Encapsulation · 9 min, including Exercise 1
The game must never store negative health through its public interface. Our current
class cannot guarantee that. Run this invalid-state example before refactoring.

In [3]:
player.hp = -1000
print("Invalid state:", player.hp)

Invalid state: -1000


A property keeps the familiar `entity.hp` syntax while routing writes through a
rule. Here we clamp health at zero and reject negative damage. `_hp` marks an
implementation detail **by convention**, not an enforced protected field.

In [3]:
class Entity:
    def __init__(self, name, hp):
        self.name = name
        self.hp = hp

    @property
    def hp(self):
        return self._hp

    @hp.setter
    def hp(self, value):
        self._hp = max(0, value)

    def take_damage(self, amount):
        if amount < 0:
            raise ValueError("Damage must be non-negative")
        self.hp -= amount

In [5]:
player = Entity("Hero", 100)
player.hp = -1000
print("Clamped health:", player.hp)
player.hp = 100
player.take_damage(150)
print("After large hit:", player.hp)

Clamped health: 0
After large hit: 0


Double-leading-underscore names use **name mangling** to reduce accidental name
collisions in subclasses. They are not security or true privacy. Inspect this
small entity's storage; the mangled attribute remains accessible.

In [2]:
class SecretEntity:
    def __init__(self):
        self.__secret = "hidden attribute"

secret = SecretEntity()
print(secret.__dict__)
print(secret._SecretEntity__secret)
secret._SecretEntity__secret = 'not-so-hidden attribute'
print(secret._SecretEntity__secret)

{'_SecretEntity__secret': 'hidden attribute'}
hidden attribute
not-so-hidden attribute


## 3. Inheritance · 5 min
`Player` and `Enemy` **are entities**: both should inherit the health rule and
`take_damage()`. They specialize that abstraction with healing and attacking.

In [8]:
class Player(Entity):
    def heal(self, amount):
        if amount < 0:
            raise ValueError("Healing must be non-negative")
        self.hp += amount

class Enemy(Entity):
    def attack(self):
        print(f"{self.name} attacks with claws")

In [9]:
player = Player("Hero", 100)
enemy = Enemy("Goblin", 30)
player.take_damage(20)
player.heal(5)
enemy.take_damage(7)
enemy.attack()
print("Health:", player.hp, enemy.hp)

Goblin attacks with claws
Health: 85 23


**Predict:** compare the exact runtime class with membership in a hierarchy.
Connect this to the lecture's `type()`, `isinstance()`, and class objects.

In [10]:
print("Exact class:", type(player).__name__)
print("Player?", isinstance(player, Player))
print("Entity?", isinstance(player, Entity))
print("Player specializes Entity?", issubclass(Player, Entity))
print("Exact type is Entity?", type(player) is Entity)

Exact class: Player
Player? True
Entity? True
Player specializes Entity? True
Exact type is Entity? False


**Discuss:** what would we duplicate if these were independent classes? A base class
supplies inherited members; a derived class adds or replaces specialized behavior.

## 4. Overriding and super() · 5 min
Enemies need a hit message while retaining the health invariant. Redefine `Enemy`
and recreate the example. **Predict:** what breaks if the delegation is removed?

In [11]:
class Enemy(Entity):
    def take_damage(self, amount):
        print(f"Enemy {self.name} takes {amount} damage")
        super().take_damage(amount)

    def attack(self):
        print(f"{self.name} attacks with claws")

In [12]:
enemy = Enemy("Goblin", 30)
enemy.take_damage(8)
print("Health:", enemy.hp)
print("Lookup order:", [cls.__name__ for cls in Enemy.__mro__])

Enemy Goblin takes 8 damage
Health: 22
Lookup order: ['Enemy', 'Entity', 'object']


**Overriding** supplies a new implementation of an inherited method. It differs
from signature-based **overloading** in Java/C++: defining the same method name
twice in one Python class replaces the earlier definition.

`super()` delegates to the **next implementation according to the MRO** (method
resolution order). Here the next `take_damage` implementation is in `Entity`;
with multiple inheritance the next implementation can be in a sibling branch.

## 5. Polymorphism · 5 min
The game turn loop should request an attack without branching on concrete types.
Keep the earlier healing behavior while adding a player attack.

In [13]:
class Player(Entity):
    def heal(self, amount):
        if amount < 0:
            raise ValueError("Healing must be non-negative")
        self.hp += amount

    def attack(self):
        print(f"{self.name} swings a sword")

class Goblin(Enemy):
    def attack(self):
        print(f"{self.name} stabs with a dagger")

class Dragon(Enemy):
    def attack(self):
        print(f"{self.name} breathes fire")

In [14]:
player = Player("Hero", 100)
goblin = Goblin("Grub", 30)
dragon = Dragon("Ember", 200)
entities = [player, goblin, dragon]
for entity in entities:
    entity.attack()

Hero swings a sword
Grub stabs with a dagger
Ember breathes fire


In [15]:
def perform_attack(attacker):
    attacker.attack()

perform_attack(dragon)

Ember breathes fire


**Predict / discuss:** swap the objects in the list. Why does the turn loop need no
changes? Polymorphism means the same operation has different implementations
selected through the runtime object. Keep this function for the next experiment.

## 6. Duck Typing · 4 min
A turret can join the turn loop without belonging to the `Entity` hierarchy.
**Predict:** can `perform_attack` use it? Is it an `Entity`?

In [16]:
class Turret:
    def attack(self):
        print("Turret fires")

perform_attack(Turret())
print("Entity?", isinstance(Turret(), Entity))

Turret fires
Entity? False


**Duck typing:** “If an object supports the operation we need, we can use it.”
The expected method must also have suitable arguments and behavior; matching a
name alone does not guarantee a useful contract.

Calling `attack()` directly is flexible. `isinstance()` is useful when nominal
membership matters, such as a boundary requiring a specific API. Neither approach
is universally better. **Discuss:** what would happen if we passed a `Position`
object with no attack method? We introduce an explicit subclass contract next.

## 7. Abstract Base Classes · 6 min
An enemy plugin author should be told when an attack implementation is missing.
Use an ABC for that explicit subclass contract. It checks abstract methods are
implemented before instantiation; it does not validate their semantics or signatures.

In [4]:
from abc import ABC, abstractmethod

class CombatEntity(Entity, ABC):
    @abstractmethod
    def attack(self):
        """Perform this entity's attack."""
        raise NotImplementedError

In [5]:
e = CombatEntity('Enemy')

TypeError: Can't instantiate abstract class CombatEntity without an implementation for abstract method 'attack'

Refactor player and enemy onto the contract. Later enemies inherit this concrete
`attack()` by default. This is a deliberate new stage of the game model.

In [19]:
class Player(CombatEntity):
    def heal(self, amount):
        if amount < 0:
            raise ValueError("Healing must be non-negative")
        self.hp += amount

    def attack(self):
        print(f"{self.name} swings a sword")

class Enemy(CombatEntity):
    def take_damage(self, amount):
        print(f"Enemy {self.name} takes {amount} damage")
        super().take_damage(amount)

    def attack(self):
        print(f"{self.name} attacks with claws")

In [20]:
player = Player("Hero", 100)
enemy = Enemy("Goblin", 30)
for attacker in (player, enemy, Turret()):
    perform_attack(attacker)
print("Player honors nominal contract?", isinstance(player, CombatEntity))
print("Turret honors nominal contract?", isinstance(Turret(), CombatEntity))

Hero swings a sword
Goblin attacks with claws
Turret fires
Player honors nominal contract? True
Turret honors nominal contract? False


**Discuss:** why can the same function still accept a turret? Duck typing describes
how the function uses objects; the ABC expresses a contract for participating
subclasses. We did not add a type gate to `perform_attack`.

## 8. Multiple Inheritance and Mixins · 6 min, including Exercise 2
An enemy may fly without making flight the enemy's entire identity. A mixin adds
one small reusable behavior; it usually is not a standalone domain object.
Here the mixin expects its host to supply `name`.

In [21]:
class FlyingMixin:
    def fly(self):
        print(f"{self.name} flies")

class FlyingEnemy(FlyingMixin, Enemy):
    pass

In [22]:
wyvern = FlyingEnemy("Skyfang", 80)
wyvern.fly()
wyvern.attack()
wyvern.take_damage(10)
print("Health:", wyvern.hp)

Skyfang flies
Skyfang attacks with claws
Enemy Skyfang takes 10 damage
Health: 70


## 9. Method Resolution Order · 5 min
**Predict:** where does `wyvern.fly` come from? Which implementation of
`take_damage` runs first? Read the order from left to right.

In [24]:
print([cls.__name__ for cls in FlyingEnemy.__mro__])
print([cls.__name__ for cls in FlyingEnemy.mro()])

['FlyingEnemy', 'FlyingMixin', 'Enemy', 'CombatEntity', 'Entity', 'ABC', 'object']
['FlyingEnemy', 'FlyingMixin', 'Enemy', 'CombatEntity', 'Entity', 'ABC', 'object']


Attribute lookup follows this deterministic order across class dictionaries
(subject to Python's descriptor and instance-attribute rules). Python uses C3
linearization, preserving local base ordering and consistent precedence. We need
the resulting order today, not the algorithm's derivation.

`super()` searches after the class containing the method in the runtime object's
MRO. A cooperative chain needs compatible signatures and each participating
implementation to delegate once, except the terminal implementation.

### OPTIONAL / IF TIME — Tiny cooperative diamond
Two enemy behaviors share a terminal action. **Predict the trace before running**;
notice that `B` delegates to `C` even though `C` is not a direct base of `B`.

In [25]:
class A:
    def action(self):
        print("A: terminal game action")

class B(A):
    def action(self):
        print("B: log game action")
        super().action()

class C(A):
    def action(self):
        print("C: animate game action")
        super().action()

class D(B, C):
    def action(self):
        print("D: start enemy action")
        super().action()

In [26]:
print([cls.__name__ for cls in D.mro()])
D().action()

['D', 'B', 'C', 'A', 'object']
D: start enemy action
B: log game action
C: animate game action
A: terminal game action


**OPTIONAL discussion:** replacing `super().action()` in `B` with `A.action(self)`
would skip `C`. Adding the direct call as well would risk duplicate work. This is
why multiple inheritance requires deliberate cooperation.

## 10. Composition vs Inheritance · 6 min
A growing hierarchy `PlayerWithSword`, `PlayerWithBow`, `PlayerWithMagicSword`,
then all their flying variants, mixes character identity with equipment choices.
A player **is a** combat entity but **has a** replaceable weapon.
Prefer composition when variation belongs to a replaceable component.

In [27]:
class Weapon(ABC):
    @abstractmethod
    def attack(self):
        raise NotImplementedError

class Sword(Weapon):
    def attack(self):
        print("Sword slashes")

class Bow(Weapon):
    def attack(self):
        print("Bow shoots an arrow")

In [28]:
class Player(CombatEntity):
    def __init__(self, name, hp, weapon):
        super().__init__(name, hp)
        self.weapon = weapon

    def heal(self, amount):
        if amount < 0:
            raise ValueError("Healing must be non-negative")
        self.hp += amount

    def attack(self):
        print(f"{self.name} uses their weapon:")
        self.weapon.attack()

In [29]:
player = Player("Hero", 100, Sword())
perform_attack(player)
player.weapon = Bow()
perform_attack(player)
print("Still an Entity?", isinstance(player, Entity))

Hero uses their weapon:
Sword slashes
Hero uses their weapon:
Bow shoots an arrow
Still an Entity? True


**Discuss:** changing equipment changes behavior without changing the player's
class. The weapon interface could also be consumed through duck typing. What
state belongs to the player, and what belongs to equipment?

## 11. Dataclasses · 6 min
The player now needs a position and an inventory. A position is value-like: we want
readable debugging and equality by coordinates. Start with the manual boilerplate.

In [30]:
class Position:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Position(x={self.x!r}, y={self.y!r})"

    def __eq__(self, other):
        if type(other) is not type(self):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

print(Position(1, 2))
print(Position(1, 2) == Position(1, 2))

Position(x=1, y=2)
True


`@dataclass` generates `__init__`, `__repr__`, and `__eq__` for declared fields.
It is still a normal Python class; type annotations do not enforce runtime types.
Defaults make spawn positions convenient. **Predict:** equality versus identity?

In [31]:
from dataclasses import dataclass, field

@dataclass
class Position:
    x: float = 0.0
    y: float = 0.0

p1, p2 = Position(1, 2), Position(1, 2)
print(p1, Position())
print("Equal:", p1 == p2, "Identical:", p1 is p2)

Position(x=1, y=2) Position(x=0.0, y=0.0)
Equal: True Identical: False


In [32]:
@dataclass
class Inventory:
    items: list[str] = field(default_factory=list)

first, second = Inventory(), Inventory()
first.items.append("potion")
print("First:", first)
print("Second:", second)
print("Same list?", first.items is second.items)
player.position = Position()
player.inventory = first
print(player.name, player.position, player.inventory)

First: Inventory(items=['potion'])
Second: Inventory(items=[])
Same list? False
Hero Position(x=0.0, y=0.0) Inventory(items=['potion'])


`default_factory=list` creates a new list per instance. `items=[]` would put a
shared mutable object in a default; dataclasses reject this list default with
`ValueError`. This connects to reference semantics: each inventory should own a
separate list. **Discuss:** what changes if we deliberately pass the same list to
two inventory constructors? Dataclasses do not copy supplied objects automatically.

## 12. Slots · 4 min
A typo in a coordinate should be visible, and a game may allocate many positions.
First inspect the ordinary instance's dictionary. **Predict:** can it grow a `z`?

In [33]:
ordinary_position = Position(1, 2)
print(ordinary_position.__dict__)
ordinary_position.z = 10
print("After adding z:", ordinary_position.__dict__)

{'x': 1, 'y': 2}
After adding z: {'x': 1, 'y': 2, 'z': 10}


In [34]:
@dataclass(slots=True)
class SlottedPosition:
    x: float = 0.0
    y: float = 0.0

p = SlottedPosition(1, 2)
p.x = 5
print(p, "Has __dict__?", hasattr(p, "__dict__"))
try:
    p.z = 10
except AttributeError as error:
    print(type(error).__name__ + ":", error)

SlottedPosition(x=5, y=2) Has __dict__? False
AttributeError: 'SlottedPosition' object has no attribute 'z'


Slots declare fixed instance storage: this class rejects undeclared attributes and
often reduces per-instance memory overhead. Attribute access may be modestly faster;
measure on the target workload. Exact effects depend on implementation/version/platform.
We do not infer total memory from a shallow `sys.getsizeof` measurement.

**Slots are not encapsulation and do not make attributes private:** `p.x` is writable.
A base class with an instance `__dict__`, or a subclass without appropriate slots,
can retain/restore dynamic attributes. Choose slots for storage constraints, not
as an automatic optimization for every class.

## 13. Class Methods and Static Methods · 3 min
A save file stores `Alice:100`. An alternative constructor should respect the
class through which it is called. Extend the current player with a factory.

In [35]:
class SavedPlayer(Player):
    @classmethod
    def from_string(cls, text):
        name, hp_text = text.rsplit(":", 1)
        return cls(name, int(hp_text), Sword())

class ElitePlayer(SavedPlayer):
    pass

loaded = SavedPlayer.from_string("Alice:100")
elite = ElitePlayer.from_string("Champion:150")
print(type(loaded).__name__, loaded.name, loaded.hp)
print(type(elite).__name__, elite.name, elite.hp)

SavedPlayer Alice 100
ElitePlayer Champion 150


**Discuss:** hard-coding `SavedPlayer(...)` would lose the subclass in the second
call. `cls` supports compatible subclasses; a subclass with an incompatible
constructor must adapt the factory.

| Method | Automatic argument | Game use |
| --- | --- | --- |
| Instance method | `self` | Damage this particular entity |
| Class method | `cls` | Construct the requested player class |
| Static method | Neither | Class-associated helper |

### OPTIONAL / IF TIME — Static name validation
A helper can logically belong beside player loading without needing an instance
or a class. A module-level function would also be reasonable.

In [36]:
class ValidatedPlayer(SavedPlayer):
    @staticmethod
    def is_valid_name(name):
        return isinstance(name, str) and bool(name.strip()) and ":" not in name

print(ValidatedPlayer.is_valid_name("Alice"))
print(ValidatedPlayer.is_valid_name("  "))

True
False


## 14. Special Methods / Protocols · 3 min
Custom objects can participate in normal Python syntax. We already saw `repr(p)`
and `p1 == p2` through `__repr__` and `__eq__`. Inventory needs one more operation:
`len(player.inventory)`. A subclass adds that operation without changing item storage.

In [37]:
class GameInventory(Inventory):
    def __len__(self):
        return len(self.items)

player.inventory = GameInventory(["potion", "key"])
print(player.inventory)
print("Item count:", len(player.inventory))

GameInventory(items=['potion', 'key'])
Item count: 2


**Predict / discuss:** how does the count change after appending an item? Why should
an API caller use `len(inventory)` rather than know its storage details?

### OPTIONAL / IF TIME — Equality contract
The manual position returned `NotImplemented` for unsupported types, allowing Python
to try the other operand's comparison. Dataclass equality compares values for
instances of the same exact class; it does not equate every object with `x` and `y`.

In [38]:
print("Position versus tuple:", Position(1, 2) == (1, 2))
print("Ordinary versus slotted:", Position(1, 2) == SlottedPosition(1, 2))

Position versus tuple: False
Ordinary versus slotted: False


## 15. __getattribute__ · 6 min, including Exercise 3
For debugging, trace entity attribute reads. Normal explicit instance attribute
lookup such as `entity.name` goes through `__getattribute__`. Attribute machinery
looks up the hook on the type. Some implicit operations, including special-method
lookup for `len(obj)`, use type-level lookup instead of this instance hook.

In [39]:
class LoggingEntity(Entity):
    def __getattribute__(self, name):
        print("Access:", name)
        return super().__getattribute__(name)

In [40]:
logged = LoggingEntity("Observer", 20)
print("Name:", logged.name)
print("Health:", logged.hp)

Access: name
Name: Observer
Access: hp
Access: _hp
Health: 20


**Predict / discuss:** why does reading `hp` also log `_hp`? The property getter
performs another lookup.

Recursion hazard — **do not run this implementation**:
```python
# def __getattribute__(self, name):
#     print(self.name)  # invokes this same hook again
```
Delegate with `super().__getattribute__(name)` or
`object.__getattribute__(self, name)` instead of reading attributes on `self`
inside the hook. Overriding this hook is powerful and usually unnecessary for
ordinary class design; prefer properties for a particular invariant.
`__getattr__` is a fallback after normal lookup raises `AttributeError`;
`__getattribute__` handles essentially every normal explicit instance lookup.

### Exercise 3 — Public Attribute Logging · 3 min within this section
Implement the hook in `PublicLoggingEntity` below. Log `Access: <name>` only when
`name` does not start with `_`. Return the real value and preserve normal missing
attribute errors. Avoid recursion.

**Check:** reading `probe.name` logs `name`; reading `probe.hp` logs `hp` but not
`_hp`; reading `probe._hp` produces no log. A missing public attribute should log
once and raise `AttributeError`. Catch that expected error in your experiment.

In [41]:
class PublicLoggingEntity(Entity):
    pass  # TODO: implement __getattribute__ here.

# TODO: create probe and try the reads specified above.

In [45]:
# TODO: your final model here. Use multiple small cells if helpful.
# TODO: run the acceptance scenario and explain your design choices.

In [46]:
# TODO: implement the two cooperative mixins and two base orderings.
# TODO: print their MROs, call attack(), and compare trace order and final hp.

## 17. Summary · 2 min
| Mechanism | Problem it solves in this game |
| --- | --- |
| Encapsulation | Keep health valid through a stable public interface |
| Inheritance | Specialize an entity abstraction |
| Overriding / super() | Add behavior and delegate along the MRO |
| Polymorphism | Run one attack loop with different implementations |
| Duck typing | Accept a turret based on supported behavior |
| ABC | State an explicit subclass contract |
| Multiple inheritance / mixin | Combine small reusable behaviors carefully |
| MRO | Determine lookup and cooperative delegation order |
| Composition | Replace equipment without replacing the player class |
| Dataclass | Reduce position/inventory data boilerplate |
| Slots | Constrain attribute storage and often reduce overhead |
| Class method | Load the requested player class |
| Static method (optional) | Group a helper that needs neither self nor cls |
| Special methods | Support repr, equality, and len |
| __getattribute__ | Intercept normal explicit instance attribute reads |

Good OOP means choosing useful abstractions, protecting invariants, and making
behavior replaceable. **Exit ticket:** explain one inheritance choice and one
composition choice in your refactoring. Which mechanism would you leave out?